## BERT4Rec Validation & Marix 추가

In [2]:
import json, random
import torch
import torch.nn as nn
import numpy as np
from torch.utils.data import Dataset, DataLoader
from tqdm import tqdm


import wandb

# --- 1) 데이터 로드 & user_seqs 생성 (기존 코드 그대로) ---
with open('./util/result.json','r') as f:
    raw_data = json.load(f)

rows = []
for uid, user in enumerate(raw_data):
    for ts, token in enumerate(user['token_sequence']):
        rows.append([uid, token, ts])
import pandas as pd
df = pd.DataFrame(rows, columns=["user_id","item_id","timestamp"])
user_seqs = df.groupby("user_id")["item_id"].apply(list).tolist()

# --- 2) 토큰 ↔ ID 매핑 (기존 코드 그대로) ---
unique_items = sorted(df["item_id"].unique().tolist())
token2id = {t:i+1 for i,t in enumerate(unique_items)}
token2id['[MASK]'] = len(token2id)+1
id2token = {v:k for k,v in token2id.items()}

# --- 3) Dataset 정의 (val 모드 지원) ---
class BERT4RecDataset(Dataset):
    def __init__(self, sequences, token2id, max_len=20, mask_ratio=0.2, val=False):
        self.sequences     = sequences
        self.token2id      = token2id
        self.max_len       = max_len
        self.mask_ratio    = mask_ratio
        self.val           = val
        self.mask_token_id = token2id['[MASK]']

    def __len__(self):
        return len(self.sequences)

    def __getitem__(self, idx):
        seq = self.sequences[idx]
        # 1) 토큰→ID, truncate, left-pad
        ids = [self.token2id[t] for t in seq if t in self.token2id]
        ids = ids[-self.max_len:]
        pad = [0]*(self.max_len - len(ids))
        input_ids = pad + ids

        labels     = [-100]*self.max_len
        masked_ids = input_ids.copy()

        if self.val:
            # validation: 마지막 non-pad 토큰만 마스크
            last_pos = max(i for i,x in enumerate(input_ids) if x!=0)
            labels[last_pos]     = input_ids[last_pos]
            masked_ids[last_pos] = self.mask_token_id

        else:
            # train: 랜덤 마스크 + 최소 1개 보장
            for i in range(self.max_len):
                if masked_ids[i]!=0 and random.random()<self.mask_ratio:
                    labels[i]     = masked_ids[i]
                    masked_ids[i] = self.mask_token_id
            if all(l==-100 for l in labels):
                last_pos = max(i for i,x in enumerate(input_ids) if x!=0)
                labels[last_pos]     = input_ids[last_pos]
                masked_ids[last_pos] = self.mask_token_id

        return (
            torch.tensor(masked_ids, dtype=torch.long),
            torch.tensor(labels,     dtype=torch.long),
        )

# --- 4) sequential hold‐out split: train on prefix, val on last ---
#    시퀀스 길이>=2인 것만 사용
valid_seqs = [seq for seq in user_seqs if len(seq)>=2]
train_seqs = [seq[:-1] for seq in valid_seqs]   # [A,B,C]
val_seqs   = valid_seqs                         # [A,B,C,D]

# --- 5) DataLoader 생성 ---
config = {
    "n_layers": 2,
    "n_heads": 2,
    "hidden_size": 64,
    "inner_size": 256,
    "hidden_dropout_prob": 0.2,
    "attn_dropout_prob": 0.2,
    "hidden_act": "gelu",
    "layer_norm_eps": 1e-12,
    "initializer_range": 0.02,
    "mask_ratio": 0.2,
    "loss_type": "CE",
    "max_seq_length": 20,
    "n_items": len(token2id)
}

wandb.init(
    project="seq_rec",                # 원하는 프로젝트 이름
    entity="ai_project_team2",        # 스크린샷에 보인 팀 이름
    name=f"bert4rec_run_{random.randint(1000,9999)}",  # 실험 이름
    config=config
)
train_ds = BERT4RecDataset(train_seqs, token2id,
                           max_len=config['max_seq_length'],
                           mask_ratio=config['mask_ratio'],
                           val=False)
val_ds   = BERT4RecDataset(val_seqs, token2id,
                           max_len=config['max_seq_length'],
                           mask_ratio=0.0,
                           val=True)

train_loader = DataLoader(train_ds, batch_size=4, shuffle=True)
val_loader   = DataLoader(val_ds,   batch_size=4, shuffle=False)

# --- 6) 모델·손실·최적화 정의 (기존 코드 그대로) ---
class BERT4Rec(nn.Module):
    def __init__(self, cfg):
        super().__init__()
        self.hidden_size = cfg['hidden_size']
        self.max_len     = cfg['max_seq_length']
        self.n_items     = cfg['n_items']

        self.item_emb     = nn.Embedding(self.n_items+2, self.hidden_size, padding_idx=0)
        self.pos_emb      = nn.Embedding(self.max_len, self.hidden_size)
        encoder_layer = nn.TransformerEncoderLayer(
            d_model=self.hidden_size, nhead=cfg['n_heads'],
            dim_feedforward=cfg['inner_size'], dropout=cfg['hidden_dropout_prob'],
            activation="gelu", layer_norm_eps=cfg['layer_norm_eps'],
            batch_first=True
        )
        self.encoder = nn.TransformerEncoder(encoder_layer, num_layers=cfg['n_layers'])
        self.norm    = nn.LayerNorm(self.hidden_size, eps=cfg['layer_norm_eps'])
        self.drop    = nn.Dropout(cfg['hidden_dropout_prob'])
        self.out     = nn.Linear(self.hidden_size, self.n_items+1)
        self._init_weights(cfg['initializer_range'])

    def _init_weights(self, std):
        for n,p in self.named_parameters():
            if 'weight' in n: nn.init.normal_(p,0,std)
            elif 'bias' in n: nn.init.constant_(p,0)

    def forward(self, input_ids):
        pos = torch.arange(self.max_len, device=input_ids.device) \
                    .unsqueeze(0).expand_as(input_ids)
        x = self.item_emb(input_ids) + self.pos_emb(pos)
        x = self.norm(x); x = self.drop(x)
        pad_mask = (input_ids==0)
        h = self.encoder(x, src_key_padding_mask=pad_mask)
        return self.out(h)

device    = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model_bert     = BERT4Rec({**config, **{"n_layers":2,"n_heads":2,"inner_size":256,
                                  "hidden_size":64,"hidden_dropout_prob":0.2,
                                  "layer_norm_eps":1e-12,"initializer_range":0.02,
                                  "n_items":len(token2id)}}).to(device)
opt       = torch.optim.Adam(model_bert.parameters(), lr=1e-3)
criterion = nn.CrossEntropyLoss(ignore_index=-100)

# --- 7) 평가 지표 헬퍼 (Precision, Recall, HR, F1 @k) ---
def evaluate_ranking(all_scores, all_labels, ks=[1,5,10]):
    N, V = all_scores.shape
    rank = np.argsort(-all_scores, axis=1)
    metrics = {}
    for k in ks:
        topk = rank[:,:k]
        hits = np.array([1 if all_labels[i] in topk[i] else 0 for i in range(N)])
        prec = hits.mean()/k
        rec  = hits.mean()   # single-ground-truth
        hr   = rec
        f1   = 2*prec*rec/(prec+rec) if (prec+rec)>0 else 0.0
        metrics.update({f"P@{k}":prec, f"R@{k}":rec, f"HR@{k}":hr, f"F1@{k}":f1})
    return metrics

# --- 8) Train + Val 루프 ---
for epoch in tqdm(range(1, 31)):
    # -- train --
    model_bert.train()
    total_loss = 0
    for x, y in train_loader:
        x, y = x.to(device), y.to(device)
        logits = model_bert(x)              # (B, L, V)
        B, L, V = logits.shape
        loss = criterion(logits.view(-1,V), y.view(-1))
        opt.zero_grad(); loss.backward(); opt.step()
        total_loss += loss.item()
    train_loss = total_loss / len(train_loader)

    # -- val --
    model_bert.eval()
    val_loss = 0
    all_scores, all_labels = [], []
    with torch.no_grad():
        for x, y in val_loader:
            x, y = x.to(device), y.to(device)
            logits = model_bert(x)
            B, L, V = logits.shape
            val_loss += criterion(logits.view(-1,V), y.view(-1)).item()
            for b in range(B):
                # 1) 마스킹된 위치 (정답이 있는 위치)
                pos = (y[b] != -100).nonzero(as_tuple=True)[0].item()
                scores = logits[b, pos, :].clone()  # (V,)

                # 2) 입력 시퀀스에서 이미 등장한 토큰들은 제외
                input_token_ids = set(x[b].tolist())
                for t_id in input_token_ids:
                    if t_id != 0:  # padding 제외
                        scores[t_id] = float('-inf')

                all_scores.append(scores.cpu().numpy())
                all_labels.append(y[b, pos].item())

    val_loss /= len(val_loader)
    metrics = evaluate_ranking(np.stack(all_scores), np.array(all_labels))

    # print(f"Epoch {epoch:02d}  Train Loss: {train_loss:.4f}  Val Loss: {val_loss:.4f}")
    # for k in [1,5,10]:
    #     print(f"  P@{k}: {metrics[f'P@{k}']:.4f}, R@{k}: {metrics[f'R@{k}']:.4f}, "
    #           f"HR@{k}: {metrics[f'HR@{k}']:.4f}, F1@{k}: {metrics[f'F1@{k}']:.4f}")
    wandb.log({
        "epoch": epoch,
        "train_loss": train_loss,
        "val_loss": val_loss,
        "P@5": metrics["P@5"],
        "R@5": metrics["R@5"],
        "F1@5": metrics["F1@5"],
        "P@10": metrics["P@10"],
        "R@10": metrics["R@10"],
        "F1@10": metrics["F1@10"],
    })


wandb: Currently logged in as: kwon04210 (listwiserank) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin


100%|██████████| 30/30 [02:03<00:00,  4.11s/it]


In [4]:
# ── 전제: model 은 학습된 BERT4Rec, token2id/id2token, device 가 정의되어 있다고 가정
max_len       = config['max_seq_length']
mask_token_id = token2id['[MASK]']
# Create 10 random sequences with lengths 8~12
random_seqs = [['CERT_IPE', 'Git', 'AWARD_UNIV', 'CERT_DATA', 'AWARD_OUTER']]
# for _ in range(10):
#     length = random.randint(8, 12)
#     seq = random.choices(unique_items, k=length)
#     random_seqs.append(seq)


model_bert.eval()
with torch.no_grad():
    for idx, seq in enumerate(random_seqs, 1):
        # 1) 토큰 → ID, truncate, left-pad
        ids = [token2id[t] for t in seq]           # Python까지 포함
        ids = ids[-(max_len-1):]                   # 공간 확보를 위해 하나 덜 잘라내고
        masked_ids = ids + [mask_token_id]         # 마지막에 [MASK]를 추가
        pad = [0] * (max_len - len(masked_ids))
        input_ids = pad + masked_ids               # Python 정보가 남아 있게 됨
        # 2) 마지막 non-pad 위치만 MASK
        masked_ids = input_ids.copy()
        masked_ids[max_len-1] = mask_token_id

        # 3) tensor로 변환
        inp = torch.tensor([masked_ids], dtype=torch.long, device=device)
        logits = model_bert(inp)

        # 4) 마스크 위치 logit 추출 후, 중복 제외
        last_logits = logits[0, max_len-1].clone()
        for t_id in set(input_ids):
            if t_id != 0:
                last_logits[t_id] = float('-inf')

        topk_ids    = torch.topk(last_logits, k=5).indices.tolist()
        topk_tokens = [id2token[i] for i in topk_ids]

        print(f"\nTest#{idx}  입력시퀀스: {seq}")
        print(f"추천 Top-5 (중복 제거): {topk_tokens}")




Test#1  입력시퀀스: ['CERT_IPE', 'Git', 'AWARD_UNIV', 'CERT_DATA', 'AWARD_OUTER']
추천 Top-5 (중복 제거): ['TYPE_Maintenance|ROLE_FE|SKILL_HTMLCSS', 'TYPE_Intern|ROLE_AI|SKILL_Python', 'TYPE_Club|ROLE_NUL|SKILL_NUL', 'TYPE_Club|ROLE_FULLSTACK|SKILL_NUL', 'TYPE_Club|ROLE_AI|SKILL_Python']


## SASRec Validation & Matrix

In [ ]:
import json
import random
import numpy as np
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from tqdm import tqdm
import wandb
import pandas as pd

# ==================== 1. Load & preprocess ====================
with open('../util/result.json','r') as f:
    raw = json.load(f)
rows = []
for uid, user in enumerate(raw):
    for ts, tok in enumerate(user['token_sequence']):
        rows.append((uid, tok, ts))
_df = pd.DataFrame(rows, columns=['user_id','item_id','timestamp'])
user_seqs = _df.groupby('user_id')['item_id'].apply(list).tolist()
valid_seqs = [seq for seq in user_seqs if len(seq) >= 2]
unique_items = sorted(_df['item_id'].unique())
token2id = {t: i+1 for i, t in enumerate(unique_items)}  # 0 = PAD
id2token = {i: t for t, i in token2id.items()}

# ==================== 2. Dataset definitions ====================
class SeqLabelDataset(Dataset):
    """
    Full-sequence dataset for predicting next item at each position
    """
    def __init__(self, sequences, t2i, max_len=20):
        self.seqs    = sequences
        self.t2i     = t2i
        self.max_len = max_len
    def __len__(self):
        return len(self.seqs)
    def __getitem__(self, idx):
        ids = [self.t2i[t] for t in self.seqs[idx] if t in self.t2i]
        ids = ids[-self.max_len:]
        L = len(ids)
        pad_len   = self.max_len - L
        input_ids = [0]*pad_len + ids
        labels    = [0]*pad_len + ids[1:] + [0]
        return (
            torch.tensor(input_ids, dtype=torch.long),
            torch.tensor(labels,    dtype=torch.long)
        )

class SASRecDataset(Dataset):
    """
    Hold-out last item for recommendation metrics
    """
    def __init__(self, sequences, t2i, max_len=20):
        self.seqs    = sequences
        self.t2i     = t2i
        self.max_len = max_len
    def __len__(self):
        return len(self.seqs)
    def __getitem__(self, idx):
        ids = [self.t2i[t] for t in self.seqs[idx] if t in self.t2i]
        ids = ids[-self.max_len:]
        prefix, target = ids[:-1], ids[-1]
        L = len(prefix)
        pad_len = self.max_len - L
        input_ids = [0]*pad_len + prefix
        return (
            torch.tensor(input_ids, dtype=torch.long),
            torch.tensor(L,         dtype=torch.long),
            torch.tensor(target,    dtype=torch.long)
        )

# ==================== 3. SASRec model ====================
class SASRec(nn.Module):
    def __init__(self, cfg):
        super().__init__()
        self.max_len     = cfg['max_seq_length']
        self.hidden_size = cfg['hidden_size']
        self.n_items     = cfg['n_items']
        self.item_emb = nn.Embedding(self.n_items+1, self.hidden_size, padding_idx=0)
        self.pos_emb  = nn.Embedding(self.max_len, self.hidden_size)
        enc = nn.TransformerEncoderLayer(
            d_model=self.hidden_size,
            nhead=cfg['n_heads'],
            dim_feedforward=cfg['inner_size'],
            dropout=cfg['hidden_dropout_prob'],
            activation=cfg['hidden_act'],
            layer_norm_eps=cfg['layer_norm_eps'],
            batch_first=True
        )
        self.encoder     = nn.TransformerEncoder(enc, num_layers=cfg['n_layers'])
        self.layer_norm  = nn.LayerNorm(self.hidden_size, eps=cfg['layer_norm_eps'])
        self.dropout     = nn.Dropout(cfg['hidden_dropout_prob'])
        self.output_bias = nn.Parameter(torch.zeros(self.n_items+1))
        self._init_weights(cfg['initializer_range'])
    def _init_weights(self, std):
        for n, p in self.named_parameters():
            if 'weight' in n:
                nn.init.normal_(p, mean=0.0, std=std)
            elif 'bias' in n:
                nn.init.constant_(p, 0.0)
    def forward(self, input_ids):
        B, L = input_ids.size()
        pos = torch.arange(L, device=input_ids.device).unsqueeze(0).expand(B, L)
        x = self.item_emb(input_ids) + self.pos_emb(pos)
        x = self.layer_norm(x)
        x = self.dropout(x)
        mask = torch.triu(torch.ones((L, L), device=x.device), diagonal=1).bool()
        h = self.encoder(x, mask=mask)
        logits = torch.matmul(h, self.item_emb.weight.t()) + self.output_bias  # (B, L, V)
        return logits

# ==================== 4. Config & W&B ====================
config = {
    'n_layers':2, 'n_heads':2, 'hidden_size':64,
    'inner_size':256, 'hidden_dropout_prob':0.5,
    'hidden_act':'gelu', 'layer_norm_eps':1e-12,
    'initializer_range':0.02,
    'max_seq_length':20,
    'n_items': len(token2id)
}
wandb.init(
    project='seq_rec', entity='ai_project_team2',
    name=f'sasrec_run_{random.randint(1000,9999)}', config=config
)

# ==================== 5. DataLoaders ====================
train_ds      = SeqLabelDataset(valid_seqs, token2id, max_len=config['max_seq_length'])
val_rec_ds    = SASRecDataset(valid_seqs, token2id, max_len=config['max_seq_length'])
val_seq_ds    = SeqLabelDataset(valid_seqs, token2id, max_len=config['max_seq_length'])
train_loader  = DataLoader(train_ds, batch_size=4, shuffle=True,  drop_last=True)
val_rec_loader= DataLoader(val_rec_ds, batch_size=4, shuffle=False)
val_seq_loader= DataLoader(val_seq_ds, batch_size=4, shuffle=False)

# ==================== 6. Loss, model, optimizer ====================
device       = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
model_sas        = SASRec(config).to(device)
criterion_ce = nn.CrossEntropyLoss(ignore_index=0)
optimizer    = torch.optim.Adam(model_sas.parameters(), lr=1e-3)

# ==================== 7. Train & Validation ====================
for epoch in range(1, 51):
    model_sas.train()
    train_loss = 0
    for input_ids, labels in tqdm(train_loader, desc='Train'):
        input_ids, labels = input_ids.to(device), labels.to(device)
        logits = model_sas(input_ids)  # (B, L, V)
        B, L, V = logits.shape
        loss = criterion_ce(logits.view(B*L, V), labels.view(B*L))

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
        train_loss += loss.item()
    train_loss /= len(train_loader)

    model_sas.eval()
    # validation loss over all tokens (same as train)
    val_loss = 0
    for input_ids, labels in tqdm(val_seq_loader, desc='Val Loss'):
        input_ids, labels = input_ids.to(device), labels.to(device)
        logits = model_sas(input_ids)
        B, L, V = logits.shape
        val_loss += criterion_ce(logits.view(B*L, V), labels.view(B*L)).item()
    val_loss /= len(val_seq_loader)

    # recommendation metrics on last item
    all_scores, all_labels = [], []
    for inp, slens, tgt in val_rec_loader:
        inp, slens, tgt = inp.to(device), slens.to(device), tgt.to(device)
        logits_full = model_sas(inp)  # (B, L, V)
        B, L, V = logits_full.shape
        for b in range(B):
            last_idx = slens[b] - 1
            scores = logits_full[b, last_idx].clone()
            seen_ids = set(inp[b, -slens[b]:].tolist())
            seen_ids.discard(tgt[b].item())
            for i in seen_ids:
                if i != 0:
                    scores[i] = float('-inf')
            all_scores.append(scores.detach().cpu().numpy())
            all_labels.append(tgt[b].item())

    metrics = evaluate_ranking(np.stack(all_scores), np.array(all_labels))

    wandb.log({
        'epoch': epoch,
        'train_loss': train_loss,
        'val_loss': val_loss,
        'P@5': metrics['P@5'], 'R@5': metrics['R@5'], 'F1@5': metrics['F1@5'],
        'P@10': metrics['P@10'], 'R@10': metrics['R@10'], 'F1@10': metrics['F1@10']
    })
    print(f"Epoch {epoch:02d} | Train: {train_loss:.4f} | Val: {val_loss:.4f} | P@5 {metrics['P@5']:.4f}")

Val Loss: 100%|██████████| 250/250 [00:00<00:00, 360.15it/s]


Epoch 01 | Train: 5.5825 | Val: 5.0540 | P@5 0.0094


Val Loss: 100%|██████████| 250/250 [00:00<00:00, 361.46it/s]


Epoch 02 | Train: 4.9626 | Val: 4.7574 | P@5 0.0042


Val Loss: 100%|██████████| 250/250 [00:00<00:00, 359.33it/s]


...

Epoch 48 | Train: 3.0331 | Val: 2.5868 | P@5 0.0042


Val Loss: 100%|██████████| 250/250 [00:00<00:00, 362.76it/s]


Epoch 49 | Train: 3.0401 | Val: 2.5607 | P@5 0.0048


Val Loss: 100%|██████████| 250/250 [00:00<00:00, 366.48it/s]


Epoch 50 | Train: 3.0157 | Val: 2.5988 | P@5 0.0050


In [ ]:
import random
import torch

# 1. 시퀀스 길이 및 아이템 사전 준비
max_len     = config['max_seq_length']
unique_items = list(token2id.keys())

# 2. 랜덤 시퀀스 생성 (길이 8~12)
random_seqs = []
for _ in range(10):
    length = random.randint(8, 12)
    seq = random.choices(unique_items, k=length)
    random_seqs.append(seq)

# 3. 모델을 평가 모드로 전환
model_sas.eval()
with torch.no_grad():
    for idx, seq in enumerate(random_seqs, 1):
        # 토큰 → ID, truncate, left-pad
        ids    = [token2id[t] for t in seq if t in token2id]
        ids    = ids[-max_len:]
        seqlen = len(ids)
        pad    = [0] * (max_len - seqlen)
        input_ids = torch.tensor([pad + ids],
                                 dtype=torch.long,
                                 device=device)  # (1, L)

        # 모델 통과 → (1, L, V)
        logits_full = model_sas(input_ids)
        last_logits = logits_full[0, seqlen-1]  # (V,)

        # 입력에 존재한 아이템 제외
        for i in set(ids):
            last_logits[i] = float('-inf')

        # Top-5 추천
        topk_ids    = torch.topk(last_logits, k=5).indices.tolist()
        topk_tokens = [id2token[i] for i in topk_ids]

        print(f"\nTest#{idx}  입력시퀀스: {seq}")
        print(f"추천 Top-5 (중복 제거): {topk_tokens}")



Test#1  입력시퀀스: ['ScikitLearn', 'TYPE_Proj|ROLE_DE|SKILL_Numpy', 'Spring', 'TYPE_Proj|ROLE_BE|SKILL_Jenkins', 'TYPE_Junior|ROLE_BE|SKILL_Docker', 'TYPE_Club|ROLE_FE|SKILL_React', 'TYPE_Proj|ROLE_AI|SKILL_PostgreSQL', 'TYPE_StartUp|ROLE_FE|SKILL_Nuxt', 'TYPE_Proj|ROLE_FE|SKILL_Redux', 'TYPE_Hackathon|ROLE_FE|SKILL_VueJS']
추천 Top-5 (중복 제거): ['TYPE_Junior|ROLE_AI|SKILL_Git', 'Python', 'TYPE_StartUp|ROLE_DEVOPS|SKILL_Docker', 'TYPE_StartUp|ROLE_DEVOPS|SKILL_Kubernetes', 'TYPE_Junior|ROLE_AI|SKILL_Pandas']

Test#2  입력시퀀스: ['TYPE_StartUp|ROLE_BE|SKILL_Express', 'TYPE_Junior|ROLE_UXUI|SKILL_NUL', 'TYPE_Proj|ROLE_AI|SKILL_TensorFlow', 'TYPE_Maintenance|ROLE_FE|SKILL_Nuxt', 'TYPE_StartUp|ROLE_AI|SKILL_AWS', 'TYPE_Junior|ROLE_AI|SKILL_SQL', 'TYPE_Proj|ROLE_DE|SKILL_Oracle', 'TYPE_StartUp|ROLE_AI|SKILL_DynamoDB', 'TYPE_StartUp|ROLE_UXUI|SKILL_NUL', 'TYPE_Junior|ROLE_BE|SKILL_Kotlin', 'TYPE_Proj|ROLE_BE|SKILL_Bash']
추천 Top-5 (중복 제거): ['HTMLCSS', 'TS', 'Redux', 'TYPE_Junior|ROLE_GAME|SKILL_NUL', 'R